# 必要ファイル読み出し

In [39]:
import urllib.request
urls = ['https://raw.githubusercontent.com/HUme-ri/The-Fricativity-of-Mid-9th-Century-South-Rhine-Franconian-d-/refs/heads/main/otf_all.txt', 'https://raw.githubusercontent.com/HUme-ri/The-Fricativity-of-Mid-9th-Century-South-Rhine-Franconian-d-/refs/heads/main/otf_for_search.txt', 'https://raw.githubusercontent.com/HUme-ri/The-Fricativity-of-Mid-9th-Century-South-Rhine-Franconian-d-/refs/heads/main/quote_index.txt']
save_names = ['otf_all.txt',                                                                                                                  'otf_for_search.txt',                                                                                                                        'quote_index.txt']
for url,name in zip(urls,save_names):
  urllib.request.urlretrieve(url, name)

# 箇所検索用

In [40]:
# 全通し番号とファイル番号・詩行番号を対応付けるディクショナリのリスト
text = 'otf_for_search.txt'
For_Search = []
with open(text) as f:
  for line in f:
    line.rstrip()
    indic = dict()
    i,j,k,l = line.split('\t')
    indic['LineNumber'] = int(i)
    indic['FileNumber'] = int(j)
    indic['VerseNumber'] = int(k) + 1
    indic['Text'] = l
    For_Search.append(indic)

# FileNumberと章を対応付けるディクショナリ
text = 'quote_index.txt'
with open(text) as f:
  File_Quote = dict()
  for line in f:
    line = line.rstrip()
    i,j = line.split('\t')
    File_Quote[int(i)] = j

# 例
print(f'通し1行目の情報 {For_Search[0]}\n収録箇所:{File_Quote[For_Search[0]['FileNumber']]}')
print(f'通し1111行目の情報 {For_Search[1110]}\n収録箇所:{File_Quote[For_Search[1110]['FileNumber']]}')

通し1行目の情報 {'LineNumber': 0, 'FileNumber': 1, 'VerseNumber': 1, 'Text': 'Lúdowig ther snéllo, thes wísduames fóllo\n'}
収録箇所:Ludw
通し1111行目の情報 {'LineNumber': 1110, 'FileNumber': 26, 'VerseNumber': 7, 'Text': 'So síe tho thar gibétotun, thie fira giéntotun\n'}
収録箇所:I-22


# 分節音分割用クラス

In [41]:
#特殊記号付きアルファベットの処理用ディクショナリ
tremas = ['ä','ï','ü','ë','ö','ÿ']
accented = ['á','í','ú','é','ó','ý']
norm = ['a','i','u','e','o','y']
vow = tremas + accented + norm
norms = norm * 3
normal_dict = dict(zip(tremas+accented,norm*2))

In [42]:
# CV分割クラス
import re
class CVsep:
  def __init__(self, graph:str):
    self.graph = graph

    #符号を剝ぎ取った文字列としてself.normalを準備
    pattern = r'|'.join(normal_dict.keys())
    self.normal = re.sub(pattern, lambda match: normal_dict[match.group()], graph)

    if self.normal[-1] in 'aeiouy': #末尾が母音の場合codaとして"0"を追加
      self.normal = self.normal + '0'
    else:
      pass

    self.v_seq_norm = re.findall('[aeiouy]+', self.normal) #一致判定はこちらを用いる
    self.v_seq_orgn = re.findall(f'[{''.join(tremas+accented+norm)}]+',self.graph) #ifAx用
    self.c_seq = re.findall('[^aeiouy]+',self.normal)

  #from_lastはアクセントがあるかどうかを確認したい音節（末尾より）、YesAndはそこに全てアクセントがあることを求めるか
  def ifAx(self, from_last, YesAnd = True):
    if type(from_last) != list: #リスト扱いにする
      from_last = [from_last]
    else:
      pass

    bools = [False] * len(from_last)
    for i,syl_num in enumerate(from_last):
      try:
        if re.match(f'[{''.join(accented)}]+', self.v_seq_orgn[-syl_num]):
          bools[i] = True
        else:
          pass
      except IndexError as e: # はみ出た場合はFalseを返す
        print(e)
        return(False)

    if YesAnd == True:
      return(all(bools))
    else:
      return(any(bools))

# テキスト成形

In [43]:
#分析前のテキスト成形に用いる関数

def trema(sents): #'ạ'などは2文字判定になるため、1文字のトレマ付き文字に置き換える
  ex_sents = sents
  import re
  rep_bef = ['ạ','ị','ụ','ẹ','ọ','ỵ']
  tremas = ['ä','ï','ü','ë','ö','ÿ']
  for i, g in enumerate(rep_bef):
    ex_sents = re.sub(g,tremas[i],ex_sents)
  return ex_sents

def subs(sents): #書記慣習による混雑を整理
  ex_sents = sents
  import re
  repc_bef = ['th','(?<!(úm|lí))ph','qu','(?<=[\t])ch','ch','u(?=[áíúéóý])','gi(?=[áíúéóý])','zuene','íuer', 'iuer', 'íuih', 'iuih', 'giilta', 'fróuun', 'ríuon', 'ríuag', 'irmúait', 'éies', 'blíuenti', 'ínouon', 'dríuon', 'scouon', 'dríua', 'gidríuon', 'scóuotun', 'alaníuaz', 'niuaz', 'bliuan', 'biscóuo']
  repc_aft = ['þ', 'pf',            'kw','kh',         'hh','w'            ,'gi0'           ,'zwene','íuwer','iuwer','íuwih','iuwih','gi0ilta','fróuwun','ríuwon','ríuwag','irmúa0it','éi0es','blíuwenti','ínouwon','dríuwon','scouwon','dríuwa','gidríuwon','scóuwotun','alaníuwaz','niuwaz','bliuwan','biscóuwo']
  for i, g in enumerate(repc_bef):
    ex_sents = re.sub(g,repc_aft[i],ex_sents)
  return ex_sents


In [44]:
read_file = './otf_all.txt'
# 関数を適用させ分析に直接用いるファイルを作る
with open(read_file) as r_f, open('Otf_for_Analysis.txt','w') as f:
  for line in r_f:
    line = line.rstrip()
    line = trema(line)
    line = subs(line)
    print(line,file=f)

In [45]:
import re
read_file = 'Otf_for_Analysis.txt'

LinePairs = [] # 詩行の組と番号を格納
with open(read_file) as f:
  full_lines = f.read().split('\n')

  num = 0
  l = 0
  while l+1 < len(full_lines):
    LinePairs.append((full_lines[l],full_lines[l+1],num))
    num += 1
    l += 3

In [46]:
P_db_deleted = [] #同一詩行を削除したものを格納
Attested = []

for p in LinePairs:
#Attestedには今まで現れた詩行が入っている
  if (CVsep(p[0]).normal,CVsep(p[1]).normal) in Attested:

    # 最終語の遡って2音節目にアクセントのある詩行を優先して残す
    s1 = CVsep(p[0].split('\t')[-1])
    s2 = CVsep(p[1].split('\t')[-1])
    if (s1.ifAx(2,True) and not(s1.ifAx(1,True))) or (s2.ifAx(2,True) and not(s2.ifAx(1,True))):
      for ind, j in enumerate(P_db_deleted):
        if (CVsep(p[0]).normal,CVsep(p[1]).normal) == (CVsep(j[0]).normal,CVsep(j[1]).normal):
          P_db_deleted[ind] = p
        else:
          pass
  else:
    P_db_deleted.append(p)
    Attested.append((CVsep(p[0]).normal,CVsep(p[1]).normal))

print(len(LinePairs),len(Attested),len(P_db_deleted))

list index out of range
list index out of range
list index out of range
list index out of range
list index out of range
list index out of range
list index out of range
list index out of range
list index out of range
list index out of range
list index out of range
list index out of range
list index out of range
list index out of range
list index out of range
7416 7311 7311


In [47]:
#最後の語のみ抽出
lw_pairs = []
for p in P_db_deleted:
  lw_pairs.append((p[0].split('\t')[-1],p[1].split('\t')[-1],p[2]))

# カウント-アソシエーション分析関数

In [48]:
# 分節音組カウント関数
def Collect(trg_ls:list, trg_num:int, trg_c:bool, same_in:bool = False) -> list:
  '''trg_lsは集計対象のペアが入ったリスト
  trg_numは後ろから幾つめの要素を数えるか
  trg_cはTrueなら子音/Falseなら母音を数える
  same_inは同一ペアも入れるか(default False)'''

  Pairs = []

  #取得する属性名を指定
  attr = ''
  if trg_c == True:
    attr = 'c_seq'
  else:
    attr = 'v_seq_norm'

  # 対応する箇所を収集
  for p in trg_ls:
    try:
      if same_in == True:
        Pairs.append([getattr(CVsep(p[0]),attr)[-trg_num], getattr(CVsep(p[1]),attr)[-trg_num]])
      else:
        if getattr(CVsep(p[0]),attr)[-trg_num] != getattr(CVsep(p[1]),attr)[-trg_num]:
          Pairs.append([getattr(CVsep(p[0]),attr)[-trg_num], getattr(CVsep(p[1]),attr)[-trg_num]])

    except IndexError:
      print(f'IndexError:{p[0]}, {p[1]}')

  print(f'length:{len(Pairs)}')

  return Pairs

In [49]:
# 整形済み組データに対しアソシエーション分析を実行する関数
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

def association_list2csv(ls, file_name = 'default', min_supp=0.01):
  '''lsは分析対象の組を収めたリスト
  min_suppは支持度の足切りライン
  file_nameはcsvのファイル名, {file_name}.csvに'''

  # ワンホットエンコーディング処理
  transactions = ls
  te = TransactionEncoder()
  te_ary = te.fit(transactions).transform(transactions)
  df = pd.DataFrame(te_ary, columns=te.columns_)

  # 共起性の高いアイテムセットの抽出
  frequent_itemsets = apriori(df, min_support=min_supp, use_colnames=True)
  frequent_itemsets.head()
  rules_df = association_rules(
      frequent_itemsets,
      metric="confidence",
      min_threshold=0.02,
  )

  # 利用するカラム名の設定
  columns = [
      "antecedents",
      "consequents",
      "support",
      "lift",
      "confidence",
  ]
  # support, lift, confidenceの順番に降順でソート
  rules_df.sort_values(by=["support","lift","confidence"], ascending=False)[columns].reset_index(drop=True).round(
      4
  ).head()
  table_df = rules_df.sort_values(by=["support","lift","confidence"], ascending=False)[columns].reset_index(drop=True).round(
      4
  )

  table_df['antecedents'] = table_df['antecedents'].apply(lambda x: list(x)[0])
  table_df['consequents'] = table_df['consequents'].apply(lambda x: list(x)[0])
  table_df.to_csv(f'./{file_name}.csv',encoding='utf-8_sig',index=False)
  return

# 環境A

In [50]:
target = Collect(lw_pairs, 1, True)

length:282


In [51]:
association_list2csv(target, '環境A')

# 環境B

In [52]:
from IPython.display import clear_output

slctd_b = []
for p in lw_pairs:
  seqs = (CVsep(p[0]),CVsep(p[1]))

  # 両語の音節数が2以上かつどちらも一致
  try:
    if all([True if seqs[0].v_seq_norm[-i] == seqs[1].v_seq_norm[-i] else False for i in (1,2)]) and (seqs[0].c_seq[-1] == seqs[1].c_seq[-1]):
      # 少なくとも一方の語の後ろから2音節目にアクセント符号がある
      if seqs[0].ifAx(2) or seqs[1].ifAx(2):
        slctd_b.append((p[0],p[1],p[2]))
  except IndexError:
    continue


In [53]:
len(slctd_b)

2550

In [54]:
target = Collect(slctd_b,2,True)

length:623


In [55]:
association_list2csv(target, '環境B')